# Gold Transformation Pipeline

Lakeflow Declarative Pipeline stage that builds analytical tables in the **serve** schema from cleaned **refined** tables.

**Prerequisite:** `ldp_silver_transformations` must complete successfully first.

**Workflow usage:** run as the second pipeline task, depending on the silver stage.


## Configuration


In [ ]:
CATALOG = "jm_databricks_learning_ws"
REFINED_SCHEMA = "refined"
SERVE_SCHEMA = "serve"

REFINED = f"{CATALOG}.{REFINED_SCHEMA}"
SERVE = f"{CATALOG}.{SERVE_SCHEMA}"


## Imports


In [ ]:
from pyspark import pipelines as dp
from pyspark.sql.functions import (
    coalesce,
    col,
    count,
    countDistinct,
    lit,
    max as spark_max,
    sum as spark_sum,
)


## `serve.supplier_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.supplier_summary",
    comment="Purchase order volume and quantity by supplier",
    table_properties={"quality": "gold", "domain": "procurement"},
)
def serve_supplier_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    suppliers = spark.read.table(f"{REFINED}.suppliers")

    return (
        purchase_orders.join(suppliers, on="supplier_id", how="inner")
        .groupBy(
            col("supplier_id"),
            col("supplier_name"),
            col("country"),
        )
        .agg(
            count(lit(1)).alias("total_purchase_orders"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )


## `serve.customer_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.customer_summary",
    comment="Sales order volume and quantity by customer",
    table_properties={"quality": "gold", "domain": "sales"},
)
def serve_customer_summary():
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    customers = spark.read.table(f"{REFINED}.customers")

    return (
        sales_orders.join(customers, on="customer_id", how="inner")
        .groupBy(
            col("customer_id"),
            col("customer_name"),
            col("country"),
        )
        .agg(
            count(lit(1)).alias("total_sales_orders"),
            spark_sum("quantity").alias("total_quantity"),
        )
    )


## `serve.inventory_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.inventory_summary",
    comment="Current stock and material coverage by warehouse",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_inventory_summary():
    inventory = spark.read.table(f"{REFINED}.inventory")

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("latest_snapshot_date"))
    current_inventory = inventory.join(
        latest_snapshot,
        inventory.snapshot_date == latest_snapshot.latest_snapshot_date,
        how="inner",
    ).drop("latest_snapshot_date")

    warehouses = spark.read.table(f"{REFINED}.warehouses")

    return (
        current_inventory.join(warehouses, on="warehouse_id", how="inner")
        .groupBy(
            col("warehouse_id"),
            col("warehouse_name"),
            col("plant_id"),
        )
        .agg(
            spark_sum("quantity").alias("current_stock"),
            countDistinct("material_id").alias("total_materials"),
        )
    )


## `serve.material_summary`


In [ ]:
@dp.materialized_view(
    name=f"{SERVE}.material_summary",
    comment="Purchased, sold, and on-hand quantities by material",
    table_properties={"quality": "gold", "domain": "inventory"},
)
def serve_material_summary():
    purchase_orders = spark.read.table(f"{REFINED}.purchase_orders")
    sales_orders = spark.read.table(f"{REFINED}.sales_orders")
    inventory = spark.read.table(f"{REFINED}.inventory")
    materials = spark.read.table(f"{REFINED}.materials")

    purchased = purchase_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("purchased_quantity"),
    )

    sold = sales_orders.groupBy("material_id").agg(
        spark_sum("quantity").alias("sold_quantity"),
    )

    latest_snapshot = inventory.agg(spark_max("snapshot_date").alias("latest_snapshot_date"))
    current_inventory = (
        inventory.join(
            latest_snapshot,
            inventory.snapshot_date == latest_snapshot.latest_snapshot_date,
            how="inner",
        )
        .groupBy("material_id")
        .agg(spark_sum("quantity").alias("current_inventory"))
    )

    return (
        materials.select("material_id", "material_name", "material_type")
        .join(purchased, on="material_id", how="left")
        .join(sold, on="material_id", how="left")
        .join(current_inventory, on="material_id", how="left")
        .select(
            "material_id",
            "material_name",
            "material_type",
            coalesce(col("purchased_quantity"), lit(0)).alias("purchased_quantity"),
            coalesce(col("sold_quantity"), lit(0)).alias("sold_quantity"),
            coalesce(col("current_inventory"), lit(0)).alias("current_inventory"),
        )
    )
